In [ ]:
# ---- evening sweep of the Bayesian factorization machines (CPU, one core): what runs and until when -----------------
# runs next to evening_sweep.ipynb (GPU) -> start both with Run All; ~5 s per draw at rank 32 on Task 3 (200 draws ~17 min)
END_TIME = "23:59"                              # overall end (today, clock of this PC); "end_at" in evening_cpu_queue.json can move it
run_task3_calendar_fields = True                # time-aware Bayesian FM: which calendar fields (3 runs)       budgets ~0.9 h
run_task3_rank = True                           # rank 16 / 64 with the best fields (2 runs)                    ~0.8 h
run_task1_rank = True                           # Task 1 Bayesian FM at rank 64 (best single Task 1 model, rank 32: 0.7272)  ~0.75 h
run_queue = True                                # then the refinement sweeps in evening_cpu_queue.json, until the end

# Evening sweep: Bayesian factorization machines (CPU)

**Run All** once, together with `evening_sweep.ipynb`. myFM samples on one CPU core, so this notebook runs alongside the GPU sweeps.

- **Task 3, time-aware Bayesian FM** (`time_bayesian_factorization_machine`): the Task 1 Bayesian FM + one-hot calendar fields of the
  timestamp (the newer temporal features of the time-aware factorization machine). Quick check (3 + 5 draws only): without calendar
  fields 0.9109, month + year 0.8905. First which fields (the time-aware FM preferred month + year; a Bayesian FM regularises every
  field through its own prior, so more fields may pay off here), then the rank.
- **Task 1, Bayesian FM** at rank 64 (the Task 1 sweep never reached its Bayesian FM stage; rank 32 gives 0.7272).

Same mechanics as `overnight_sweep.ipynb`; log: `output/evening_bayesian_fm_log.txt`.

In [ ]:
import json
import time
import traceback
from datetime import datetime, timedelta

import torch

from grid_search import OUTPUT_DIR, SWEEP_DIR, best_values_of, keep_computer_awake, print_sweep_overview, run_grid_search

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

QUEUE_PATH = SWEEP_DIR / "evening_cpu_queue.json"
LOG_PATH = OUTPUT_DIR / "evening_bayesian_fm_log.txt"
NO_RESULT = {"best": None}                      # what a skipped or failed sweep returns (best_values_of -> {})

sweep_start = datetime.now()
end_hour, end_minute = map(int, END_TIME.split(":"))
overall_end = sweep_start.replace(hour=end_hour, minute=end_minute, second=0, microsecond=0)
if overall_end <= sweep_start:                  # END_TIME already passed today -> tomorrow
    overall_end += timedelta(days=1)
sweep_results = {}                              # sweep label -> report (summary in the last cell)


def write_log(line: str):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with open(LOG_PATH, "a", encoding="utf-8") as log_file:
        log_file.write(f"{datetime.now():%Y-%m-%d %H:%M:%S} {line}\n")


# one sweep with a time budget: no new run after budget_minutes or at the overall end, combinations in random order
# (a cut sweep still covers the whole grid); a failing sweep is printed and skipped, the next one starts
def run_timeboxed_sweep(label: str, task_name: str, model_name: str, parameter_grid: dict, budget_minutes: float, **options) -> dict:
    now = datetime.now()
    if now >= overall_end - timedelta(minutes=1):
        print(f"\nskipped {label}: overall end {overall_end:%H:%M} reached", flush=True)
        write_log(f"skipped  {label} (overall end reached)")
        return NO_RESULT
    deadline = min(now + timedelta(minutes=budget_minutes), overall_end)
    print(f"\n===== {label} | budget {budget_minutes:.0f} min -> no new run after {deadline:%H:%M} =====", flush=True)
    write_log(f"start    {label} | {task_name} / {model_name} | until {deadline:%H:%M}")
    try:
        report = run_grid_search(task_name, model_name, parameter_grid, deadline=deadline, random_order_seed=0, **options)
    except Exception as error:
        traceback.print_exc()
        print(f"{label} failed -> continuing with the next sweep", flush=True)
        write_log(f"FAILED   {label}: {type(error).__name__}: {error}"[:400])
        return NO_RESULT
    finally:
        keep_computer_awake(True)               # run_grid_search lets the computer sleep again at its end -> hold it until the end

    sweep_results[label] = report
    output_name = f"{task_name}/{model_name}_{report['started'].replace(' ', '_').replace(':', '-')}.json"
    best = report["best"]
    write_log(f"finished {label} | {report['status']} | {report['runs_finished']} runs | "
              + (f"best {best['score']:.5f} with {json.dumps(best['swept_values'])}" if best else "no run finished")
              + f" | {output_name}")
    return report


# best values of a time-aware bias model sweep -> the same settings inside timeSVD++ (its config nests the time bias model)
def as_time_svdpp_biases(time_bias_model_values: dict) -> dict:
    return {f"time_biases.{name}": value for name, value in time_bias_model_values.items()}


keep_computer_awake(True)
write_log(f"sweep started, overall end {overall_end:%Y-%m-%d %H:%M}")
print(f"start {sweep_start:%H:%M} | overall end {overall_end:%Y-%m-%d %H:%M} | log: {LOG_PATH}")

## Task 3 - time-aware Bayesian factorization machine

In [ ]:
if run_task3_calendar_fields:
    # which calendar fields (3 runs, ~17 min each)
    time_bayesian_fm_fields_sweep = run_timeboxed_sweep("T3 time Bayesian FM 1 calendar fields", "task3",
                                                        "time_bayesian_factorization_machine", {
        "calendar_fields": [("month", "year"), ("weekday", "month", "year"), ("hour_of_day", "weekday", "month", "year")],
    }, budget_minutes=55)
    time_bayesian_fm_best_values = best_values_of(time_bayesian_fm_fields_sweep)
else:
    time_bayesian_fm_best_values = {}

if run_task3_rank:
    # rank 16 / 64 with the best fields (rank 32 is in stage 1; time ~ proportional to the rank)
    time_bayesian_fm_rank_sweep = run_timeboxed_sweep("T3 time Bayesian FM 2 rank", "task3", "time_bayesian_factorization_machine", {
        **time_bayesian_fm_best_values,
        "rank": [16, 64],
    }, budget_minutes=50)

## Task 1 - Bayesian factorization machine at rank 64

In [ ]:
if run_task1_rank:
    task1_bayesian_fm_rank_sweep = run_timeboxed_sweep("T1 Bayesian FM rank 64", "task1", "bayesian_factorization_machine", {
        "rank": 64,
    }, budget_minutes=45)

## Refinement queue
Runs every entry of the queue file it has not run yet (by `name`), re-reading the file every minute, until the overall end.

In [ ]:
def read_queue() -> dict:
    try:
        return json.loads(QUEUE_PATH.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError) as error:
        print(f"queue file not readable ({type(error).__name__}: {error}) -> retry in a minute", flush=True)
        return {"sweeps": []}


# JSON has no tuples: a list inside a list ([[...], [...]]) = values to sweep, each one a tuple setting (e.g. calendar fields)
def grid_from_json(grid: dict) -> dict:
    return {name: ([tuple(value) if isinstance(value, list) else value for value in values]
                   if isinstance(values, list) and any(isinstance(value, list) for value in values) else values)
            for name, values in grid.items()}


if run_queue:
    finished_queue_entries = set()
    while True:
        queue = read_queue()
        if queue.get("end_at"):                 # the queue file can move the overall end
            overall_end = datetime.strptime(queue["end_at"], "%Y-%m-%d %H:%M")
        if datetime.now() >= overall_end - timedelta(minutes=1):
            print(f"overall end {overall_end:%H:%M} reached -> queue stopped", flush=True)
            break
        pending_entries = [entry for entry in queue.get("sweeps", []) if entry["name"] not in finished_queue_entries]
        if not pending_entries:
            time.sleep(60)
            continue
        entry = pending_entries[0]
        finished_queue_entries.add(entry["name"])
        run_timeboxed_sweep(f"queue: {entry['name']}", entry["task"], entry["model"], grid_from_json(entry["grid"]),
                            entry.get("budget_minutes", 20), **entry.get("options", {}))

## Summary
One line per sweep of this notebook (also in the log file), then every sweep file of the tasks.

In [ ]:
for label, report in sweep_results.items():
    best = report["best"]
    best_text = f"{best['score']:.5f} with {best['swept_values']}" if best else "no finished run"
    print(f"{label:<44} {report['status']:<24} {report['runs_finished']:>9} | best {best_text}")

for task_name in ('task3', 'task1'):
    print(f"\n---- {task_name} ----")
    print_sweep_overview(task_name)

keep_computer_awake(False)
write_log("sweep finished")